# Basic RAG Query Engine - Step by Step

[30-second quickstart](https://developers.llamaindex.ai/python/framework/#30-second-quickstart)

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG) application** using vector search, help you understand and implement a basic RAG system step by step:

**Install**

In [ ]:
pip install langchain langchain-community langchain-chroma chromadb sentence-transformers python-dotenv 

**Environment**

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

from colorama import Fore, Style

# Load environment variables from .env file
load_dotenv(override=True)  

# Set OPENAI_API_KEY from environment variable
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")  
if not os.environ["OPENAI_API_KEY"]:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key:")
else:
    print(f"{Fore.GREEN}OpenAI API key loaded from environment variable.{Style.RESET_ALL}")

**Initialize model**

In [ ]:
from langchain.chat_models import init_chat_model


# -----------------------------
# LLM Setup
# -----------------------------

# A low-temperature model gives more repeatable demonstrations.
# explain
model = init_chat_model(
    "openai:gpt-4.1-mini",
    temperature=0,
    timeout=30,
    max_retries=3,
)

**Data Loading**

In [ ]:
!mkdir -p 'data/forge/'
!wget \
  "https://raw.githubusercontent.com/magit/forge/refs/heads/main/CHANGELOG" \
  -O "data/forge/CHANGELOG.txt"

**Document Loading**

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader(
    "data/forge/CHANGELOG.txt",
    encoding="utf-8",
)

documents = loader.load()

**Documents Processing**

In [ ]:

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=60,
)

chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print(f"Documents loaded: {len(documents)}")
print(f"Chunks created: {len(chunks)}")
print(chunks[0].metadata)
print(documents[0].metadata)
print(documents[0].page_content[:300])



**Prompts**

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough




prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a general knowledge assistant.

Answer using only your existing knowledge.

Rules:
- Do not claim to have read local files.
- Do not invent details.
- If the answer depends on a specific document you cannot access,
  explicitly say that you do not have enough information.
- Keep the answer concise.""",
        ),
        ("human", "{question}"),
    ]
)

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a grounded knowledge assistant.

Use only the retrieved context to answer the question.

Rules:
- Cite supporting sources using their filenames in square brackets.
- Do not use outside knowledge.
- Do not invent facts.
- If the answer is not in the context, say so.
- Keep the answer focused on the question.""",
        ),
        (
            "human",
            """Retrieved context:

{context}

Question:
{question}""",
        ),
    ]
)


**utils**

In [ ]:

def format_documents(docs: list[Document]) -> str:
    """Format retrieved documents with visible source labels."""

    if not docs:
        return "No relevant context was retrieved."

    formatted = []

    for position, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source", "unknown")
        year = doc.metadata.get("year", "unknown")
        topic = doc.metadata.get("topic", "unknown")

        formatted.append(
            f"[Source {position}]\n"
            f"File: {source}\n"
            f"Year: {year}\n"
            f"Topic: {topic}\n"
            f"Content: {doc.page_content}"
        )

    return "\n\n".join(formatted)

In [ ]:

basic_chain = (
    {
        "question": RunnablePassthrough(),
    }
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:

# build a basic RAG chain that retrieves context and uses it to answer questions

In [ ]:
questions = [
    (
        "What pagination-related problem is described in the unreleased "
        "section of the Forge changelog?"
    ),
    (
        "What external platform changed the format of label IDs?"
    ),
    (
        "What operation could be completely prevented by the pagination bug?"
    ),
]

**Run without RAG**

In [ ]:
answer = basic_chain.invoke({"question": "What pagination-related problem is described in the unreleased"})
print(f"{Fore.CYAN}\nANSWER WITHOUT RAG\n{Style.RESET_ALL}")
print(f"{Fore.MAGENTA}{answer}{Style.RESET_ALL}")

print(Style.RESET_ALL)

**Run with RAG**

In [ ]:
# answer = rag_chain.invoke("What pagination-related problem is described in the unreleased")
# print(f"{Fore.CYAN}\nANSWER WITH RAG\n{Style.RESET_ALL}")
# print(f"{Fore.GREEN}{answer}{Style.RESET_ALL}")

# print(Style.RESET_ALL)